# `f_large` sweep sensitivity report

This notebook scans **all CZ sweep folders** under `/data/zp_sweeps/results`, including the newer EC2, EJ2, and EL2 folders with descriptive suffixes. It converts `f_large` from log10 gate error to linear gate error (`10**f_large`), calculates mean and median, and compares them with the nominal centre point.

For a 21-point sweep, nominal is zero-based index 10. Only rows marked `done=True` with finite `f_large` enter the statistics.


In [1]:
from pathlib import Path
import csv
import math
import re
import statistics
from html import escape
from IPython.display import HTML, display

# ---- Configuration ----
# RESULTS_DIR = Path('/data/zp_sweeps/results')
RESULTS_DIR = Path('/data/zp_sweeps/trace_decreasing')

COLUMN = 'f_large'
INCLUDE_REOPT = True      # True loads every folder, including *_reopt
TG_FILTER = None          # e.g. 91 or 182; None includes every tg
NAME_FILTER = None        # e.g. r'EC2|EJ2|EL2'; None includes every parameter
EXPORT_CSV = Path('/home/zlqed/two_qubit_gate/zeropi/two_qubit_gate_clean/cz/f_large_sensitivity_summary.csv')

assert RESULTS_DIR.is_dir(), f'Not found: {RESULTS_DIR}'


In [2]:
TRUE_VALUES = {'true', '1', 'yes', 'y', 't'}
# The suffix accepts names such as _reopt and _full_tg20_detune_near.
NAME_RE = re.compile(
    r'^cz_sweep_(?P<sweep>.+?)_tg(?P<tg>[^_]+)_(?P<timestamp>\d{8}_\d{6})(?P<suffix>.*)$'
)

def as_float(value):
    try:
        x = float(value)
        return x if math.isfinite(x) else None
    except (TypeError, ValueError):
        return None

def discover_csvs(root):
    files = []
    for folder in sorted(x for x in root.iterdir() if x.is_dir()):
        m = NAME_RE.fullmatch(folder.name)
        if not m:
            continue
        info = m.groupdict()
        is_reopt = 'reopt' in info['suffix'].lower()
        if is_reopt and not INCLUDE_REOPT:
            continue
        if TG_FILTER is not None and info['tg'] != str(TG_FILTER):
            continue
        if NAME_FILTER and not re.search(NAME_FILTER, info['sweep'], re.I):
            continue

        # Usually the CSV matches the folder name. Reopt folders may contain a
        # CSV whose filename omits the _reopt suffix, so fall back to *.csv.
        csv_path = folder / f'{folder.name}.csv'
        if not csv_path.is_file():
            matches = sorted(folder.glob('*.csv'))
            if len(matches) != 1:
                print(f'SKIP {folder}: expected one CSV, found {len(matches)}')
                continue
            csv_path = matches[0]
        # The stable folder index follows alphabetical directory order.
        info['folder_name'] = folder.name
        info['folder_index'] = len(files)
        files.append((csv_path, info, is_reopt))
    return files

files = discover_csvs(RESULTS_DIR)
print(f'Found {len(files)} CSV files in all matching result folders:')
for path, info, is_reopt in files:
    label = ' [reopt]' if is_reopt else ''
    print(f"  [{info['folder_index']:02d}] {info['sweep']} tg={info['tg']}{label}: {info['folder_name']}")


Found 16 CSV files in all matching result folders:
  [00] EC2 tg=182: cz_sweep_EC2_tg182_20260715_142026_full_tg20_detune_near_trace_decreasing
  [01] EC2 tg=91 [reopt]: cz_sweep_EC2_tg91_20260704_015433_reopt_trace_decreasing
  [02] ECJ2 tg=182 [reopt]: cz_sweep_ECJ2_tg182_20260703_144947_reopt_trace_decreasing
  [03] ECJ2 tg=91 [reopt]: cz_sweep_ECJ2_tg91_20260703_144814_reopt_trace_decreasing
  [04] EJ2 tg=182: cz_sweep_EJ2_tg182_20260715_142026_full_tg20_detune_near_trace_decreasing
  [05] EJ2 tg=91 [reopt]: cz_sweep_EJ2_tg91_20260704_015449_reopt_trace_decreasing
  [06] EL2 tg=182: cz_sweep_EL2_tg182_20260715_142105_full_tg20_detune_near_trace_decreasing
  [07] EL2 tg=91 [reopt]: cz_sweep_EL2_tg91_20260704_015457_reopt_trace_decreasing
  [08] Ec0 tg=182 [reopt]: cz_sweep_Ec0_tg182_20260704_015430_reopt_trace_decreasing
  [09] Ec0 tg=91 [reopt]: cz_sweep_Ec0_tg91_20260704_015426_reopt_trace_decreasing
  [10] Ecc tg=182 [reopt]: cz_sweep_Ecc_tg182_20260704_015445_reopt_trace_decreas

In [3]:
# List only the first two folder-name kinds: names ending in "reopt" or "near".
# Their folder_index values are the same indices used in the report tables.
reopt_or_near_folders = [
    (info['folder_index'], info['folder_name'])
    for _, info, _ in files
    if info['folder_name'].endswith(('_reopt', '_near'))
]

print(f'Folders ending in reopt or near: {len(reopt_or_near_folders)}')
for folder_index, folder_name in reopt_or_near_folders:
    print(f'  [{folder_index:02d}] {folder_name}')


Folders ending in reopt or near: 0


In [4]:
def summarize_file(path, info, is_reopt=False):
    with path.open(newline='', encoding='utf-8-sig') as fh:
        reader = csv.DictReader(fh)
        if not reader.fieldnames or COLUMN not in reader.fieldnames:
            raise KeyError(f'{path}: missing column {COLUMN!r}')
        rows = list(reader)

    n_total = len(rows)
    if n_total == 0:
        raise ValueError(f'{path}: empty CSV')

    # Generic centre rule: 21 -> index 10. Even-length sweeps are ambiguous.
    nominal_index = n_total // 2
    if n_total % 2 == 0:
        print(f'WARNING: {path.name} has {n_total} rows; using upper centre index {nominal_index}.')

    nominal_log = as_float(rows[nominal_index].get(COLUMN))
    if nominal_log is None:
        raise ValueError(f'{path}: nominal {COLUMN} at index {nominal_index} is not finite')

    valid_logs = []
    for row in rows:
        done = str(row.get('done', 'True')).strip().lower() in TRUE_VALUES
        value = as_float(row.get(COLUMN))
        if done and value is not None:
            valid_logs.append(value)

    if not valid_logs:
        raise ValueError(f'{path}: no completed rows with finite {COLUMN}')

    errors = [10.0 ** x for x in valid_logs]
    nominal_error = 10.0 ** nominal_log
    mean_error = statistics.fmean(errors)
    median_error = statistics.median(errors)
    mean_ratio = mean_error / nominal_error
    median_ratio = median_error / nominal_error

    return {
        'folder_index': info['folder_index'],
        'folder_name': info['folder_name'],
        'sweep': info['sweep'], 'tg': info['tg'], 'reopt': is_reopt,
        'points': f'{len(valid_logs)}/{n_total}',
        'valid_points': len(valid_logs), 'total_points': n_total,
        'nominal_index': nominal_index,
        'nominal_f_large': nominal_log, 'nominal_error': nominal_error,
        'mean_f_large': statistics.fmean(valid_logs),
        'median_f_large': statistics.median(valid_logs),
        'mean_error': mean_error, 'median_error': median_error,
        'mean_over_nominal': mean_ratio, 'median_over_nominal': median_ratio,
        'mean_degradation_decades': math.log10(mean_ratio),
        'median_degradation_decades': math.log10(median_ratio),
        'path': str(path),
    }

summaries, failures = [], []
for path, info, is_reopt in files:
    try:
        summaries.append(summarize_file(path, info, is_reopt))
    except Exception as exc:
        failures.append((str(path), str(exc)))

summaries.sort(key=lambda r: r['folder_index'])
print(f'Calculated {len(summaries)} files; skipped {len(failures)}')
for path, message in failures:
    print(f'  SKIP {path}: {message}')


Calculated 16 files; skipped 0


In [5]:
CSS = """
<style>
.sens-wrap {background:#181a1b; color:#d4d4d4; padding:18px 22px; border-radius:7px;
            font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace; overflow-x:auto}
.sens-wrap h3 {margin:0 0 14px; font-size:18px; font-weight:500}
.sens-wrap h4 {margin:22px 0 8px; font-size:17px; font-weight:500}
table.sens {border-collapse:collapse; font-size:15px; min-width:800px}
table.sens th {color:#ffd6ad; border-bottom:3px solid #777; padding:7px 14px; text-align:right}
table.sens td {border-bottom:1px solid #666; padding:8px 14px; text-align:right}
table.sens th:first-child, table.sens td:first-child {text-align:left}
.sens-note {color:#aaa; font-size:12px; margin-top:14px}
</style>
"""

def fmt_sci(x): return f'{x:.3e}'
def make_table(headers, rows):
    head = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    return f'<table class="sens"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'

def render_report(items):
    main_rows = [[
        r['folder_index'], r['folder_name'], r['points'], fmt_sci(r['nominal_error']),
        fmt_sci(r['mean_error']), fmt_sci(r['median_error']),
        f"{r['mean_over_nominal']:.2f}×", f"{r['median_over_nominal']:.2f}×"
    ] for r in items]
    decade_rows = [[
        r['folder_index'], r['folder_name'],
        f"{r['mean_degradation_decades']:.3f} decades",
        f"{r['median_degradation_decades']:.3f} decades"
    ] for r in items]
    log_rows = [[
        r['folder_index'], r['folder_name'], f"{r['nominal_f_large']:.6g}",
        f"{r['mean_f_large']:.6g}", f"{r['median_f_large']:.6g}"
    ] for r in items]
    html = CSS + '<div class="sens-wrap">'
    html += '<h3>Mean and median gate errors over completed sweep points</h3>'
    html += make_table(['Index', 'Folder', 'Points', 'Nominal error', 'Mean error', 'Median error',
                        'Mean/nominal', 'Median/nominal'], main_rows)
    html += '<h4>In decade terms</h4>'
    html += make_table(['Index', 'Folder', 'Mean degradation', 'Median degradation'], decade_rows)
    html += f'<h4>Literal {escape(COLUMN)} statistics (log10 error)</h4>'
    html += make_table(['Index', 'Folder', 'Nominal f_large', 'Mean f_large', 'Median f_large'], log_rows)
    html += '<div class="sens-note">Gate error = 10<sup>f_large</sup>. Nominal row = len(rows)//2 (index 10 for 21 rows).</div></div>'
    return HTML(html)

display(render_report(summaries))


Index,Folder,Points,Nominal error,Mean error,Median error,Mean/nominal,Median/nominal
0,cz_sweep_EC2_tg182_20260715_142026_full_tg20_detune_near_trace_decreasing,21/21,4.606e-05,9.335e-05,8.073e-05,2.03×,1.75×
1,cz_sweep_EC2_tg91_20260704_015433_reopt_trace_decreasing,21/21,1.590e-03,6.665e-03,4.424e-03,4.19×,2.78×
2,cz_sweep_ECJ2_tg182_20260703_144947_reopt_trace_decreasing,21/21,5.278e-05,5.317e-05,5.278e-05,1.01×,1.00×
3,cz_sweep_ECJ2_tg91_20260703_144814_reopt_trace_decreasing,21/21,1.592e-03,1.619e-03,1.592e-03,1.02×,1.00×
4,cz_sweep_EJ2_tg182_20260715_142026_full_tg20_detune_near_trace_decreasing,21/21,4.613e-05,1.525e-04,1.001e-04,3.31×,2.17×
5,cz_sweep_EJ2_tg91_20260704_015449_reopt_trace_decreasing,21/21,1.588e-03,1.126e-02,5.623e-03,7.09×,3.54×
6,cz_sweep_EL2_tg182_20260715_142105_full_tg20_detune_near_trace_decreasing,21/21,4.617e-05,6.446e-05,4.617e-05,1.40×,1.00×
7,cz_sweep_EL2_tg91_20260704_015457_reopt_trace_decreasing,21/21,1.590e-03,4.898e-03,2.777e-03,3.08×,1.75×
8,cz_sweep_Ec0_tg182_20260704_015430_reopt_trace_decreasing,21/21,5.283e-05,5.450e-05,5.283e-05,1.03×,1.00×
9,cz_sweep_Ec0_tg91_20260704_015426_reopt_trace_decreasing,21/21,1.590e-03,1.613e-03,1.590e-03,1.01×,1.00×


In [6]:
6.853-3.443-3.41

-4.440892098500626e-16

In [7]:
# Export a machine-readable summary without requiring pandas.
fieldnames = list(summaries[0]) if summaries else []
if fieldnames:
    with EXPORT_CSV.open('w', newline='', encoding='utf-8') as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summaries)
    print(f'Wrote: {EXPORT_CSV}')
else:
    print('Nothing to export.')


Wrote: /home/zlqed/two_qubit_gate/zeropi/two_qubit_gate_clean/cz/f_large_sensitivity_summary.csv


## Notes

- The notebook scans all matching `cz_sweep_*` directories in `/data/zp_sweeps/results`.
- Suffixed directories such as `_full_tg20_detune_near` are included.
- `INCLUDE_REOPT=True` also includes `_reopt` folders; set it to `False` to exclude them.
- `f_large` is a base-10 logarithm: gate error = `10**f_large`.
- Mean and median are calculated from linear gate errors; the final table separately reports literal `f_large` statistics.
- Nominal is always the centre row, `len(rows)//2` (index 10 for a 21-point sweep).

- Every discovered folder receives a zero-based `folder_index`. The same index and full folder name appear in every report table and in the exported CSV.
